# 🤖 Colab Remote Control Agent

Run all cells in this notebook. It will:
1. Prompt you for a secret API token and your ngrok auth token.
2. Install `flask` and `pyngrok`.
3. Start a tiny REST server inside Colab and expose it via ngrok.

Copy the **Public URL** and use the same API secret in your local MCP server environment variables (`COLAB_NGROK_URL` and `COLAB_API_SECRET`).

In [ ]:
# ✅ Configuration (prompts for secrets so they are not stored in the notebook)
import getpass
import os

agent_secret = getpass.getpass('Enter a strong API secret for the MCP agent: ')
if not agent_secret:
    raise ValueError('API secret is required')
os.environ['COLAB_AGENT_SECRET'] = agent_secret

ngrok_token = getpass.getpass('Enter your ngrok auth token (press Enter if already configured globally): ')
if ngrok_token:
    os.environ['COLAB_NGROK_TOKEN'] = ngrok_token

In [ ]:
# 📦 Install dependencies
# flask + pyngrok: the agent itself
# diffusers stack: image generation (king2-image / Mage-Flow)
!pip install -q flask pyngrok
!pip install -q -U diffusers transformers accelerate safetensors peft sentencepiece

In [ ]:
# 🚀 Start the agent
import base64
import hmac
import os
import io
import subprocess
import threading
import time
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

API_SECRET = os.environ.get('COLAB_AGENT_SECRET', '')
PORT = 5000
MAX_OUTPUT_CHARS = 100_000
MAX_FILE_SIZE = 5 * 1024 * 1024   # 5 MB (text files)
MAX_BINARY_SIZE = 30 * 1024 * 1024  # 30 MB (binary downloads, e.g. images)

# ---------------------------------------------------------------------------
# Image generation models
# ---------------------------------------------------------------------------
# 'king2'    : RASHID778/king2-image — SDXL LoRA (Arabic/English art model)
# 'mage-flow': microsoft/Mage-Flow — 4.1B rectified-flow text-to-image
IMAGE_MODELS = ('king2', 'mage-flow')
GENERATED_DIR = Path('/content/generated')

# Only one pipeline is kept in VRAM at a time; switching models frees the old one.
_pipeline_cache = {'name': None, 'pipe': None}
_pipeline_lock = threading.Lock()


def _get_pipeline(model_name):
    import torch
    from diffusers import DiffusionPipeline, StableDiffusionXLPipeline

    if _pipeline_cache['name'] == model_name:
        return _pipeline_cache['pipe']

    if _pipeline_cache['pipe'] is not None:
        _pipeline_cache['pipe'] = None
        _pipeline_cache['name'] = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if device == 'cuda' else torch.float32

    if model_name == 'king2':
        pipe = StableDiffusionXLPipeline.from_pretrained(
            'stabilityai/stable-diffusion-xl-base-1.0',
            torch_dtype=dtype,
            variant='fp16' if device == 'cuda' else None,
            use_safetensors=True,
        )
        pipe.load_lora_weights('RASHID778/king2-image')
    elif model_name == 'mage-flow':
        pipe = DiffusionPipeline.from_pretrained(
            'microsoft/Mage-Flow',
            torch_dtype=dtype,
        )
    else:
        raise ValueError(f'Unknown model: {model_name!r}. Use one of {IMAGE_MODELS}.')

    pipe = pipe.to(device)
    _pipeline_cache['name'] = model_name
    _pipeline_cache['pipe'] = pipe
    return pipe


def _check_auth(request):
    auth = request.headers.get('Authorization', '')
    if not auth.startswith('Bearer '):
        return False
    return hmac.compare_digest(auth.split(' ', 1)[1], API_SECRET)


def _capture_exec(code, globals_dict):
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    result = {'status': 'success', 'output': '', 'error': ''}
    try:
        with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
            exec(code, globals_dict)
        result['output'] = stdout_capture.getvalue()
        if stderr_capture.getvalue():
            result['output'] += '\n[stderr]\n' + stderr_capture.getvalue()
    except Exception as e:
        result['status'] = 'error'
        result['error'] = f'{type(e).__name__}: {e}'

    if len(result['output']) > MAX_OUTPUT_CHARS:
        result['output'] = result['output'][:MAX_OUTPUT_CHARS] + '\n... [truncated]'
    return result


@app.route('/health', methods=['GET', 'POST'])
def health():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401
    return jsonify({'status': 'ok'})


@app.route('/execute', methods=['POST'])
def execute():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    code = data.get('code', '')
    command = data.get('command', '')

    if code:
        return jsonify(_capture_exec(code, globals()))
    elif command:
        try:
            proc = subprocess.run(
                command, shell=True, capture_output=True, text=True, timeout=120
            )
            output = proc.stdout
            if proc.stderr:
                output += '\n[stderr]\n' + proc.stderr
            if len(output) > MAX_OUTPUT_CHARS:
                output = output[:MAX_OUTPUT_CHARS] + '\n... [truncated]'
            return jsonify({'status': 'success', 'output': output})
        except Exception as e:
            return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})
    else:
        return jsonify({'status': 'error', 'error': 'No code or command provided'}), 400


@app.route('/generate_image', methods=['POST'])
def generate_image():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    prompt = data.get('prompt', '')
    if not prompt:
        return jsonify({'status': 'error', 'error': 'Prompt is required'}), 400

    model = data.get('model', 'king2')
    if model not in IMAGE_MODELS:
        return jsonify(
            {'status': 'error', 'error': f'Unknown model {model!r}. Use one of {IMAGE_MODELS}.'}
        ), 400

    negative_prompt = data.get('negative_prompt', '')
    steps = int(data.get('steps', 30))
    width = int(data.get('width', 1024))
    height = int(data.get('height', 1024))
    guidance_scale = float(data.get('guidance_scale', 7.5))
    seed = data.get('seed', -1)

    try:
        import torch

        with _pipeline_lock:
            pipe = _get_pipeline(model)

            generator = None
            if seed is not None and int(seed) >= 0:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                generator = torch.Generator(device=device).manual_seed(int(seed))

            kwargs = {
                'prompt': prompt,
                'num_inference_steps': steps,
                'width': width,
                'height': height,
                'guidance_scale': guidance_scale,
            }
            if negative_prompt:
                kwargs['negative_prompt'] = negative_prompt
            if generator is not None:
                kwargs['generator'] = generator

            image = pipe(**kwargs).images[0]

        GENERATED_DIR.mkdir(parents=True, exist_ok=True)
        filename = f'{model}_{int(time.time())}.png'
        out_path = GENERATED_DIR / filename
        image.save(out_path)

        return jsonify({
            'status': 'success',
            'path': str(out_path),
            'model': model,
            'width': width,
            'height': height,
        })
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/download_file', methods=['POST'])
def download_file():
    """Return a file (e.g. a generated image) as base64."""
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        file_path = Path(path)
        if not file_path.exists():
            return jsonify({'status': 'error', 'error': 'File not found'}), 404
        if file_path.stat().st_size > MAX_BINARY_SIZE:
            return jsonify({'status': 'error', 'error': 'File too large to download'}), 400
        content_b64 = base64.b64encode(file_path.read_bytes()).decode('ascii')
        return jsonify({'status': 'success', 'content_b64': content_b64, 'size': file_path.stat().st_size})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/read_file', methods=['POST'])
def read_file():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        file_path = Path(path)
        if not file_path.exists():
            return jsonify({'status': 'error', 'error': 'File not found'}), 404
        if file_path.stat().st_size > MAX_FILE_SIZE:
            return jsonify({'status': 'error', 'error': 'File too large to read'}), 400
        content = file_path.read_text(encoding='utf-8')
        return jsonify({'status': 'success', 'content': content})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/write_file', methods=['POST'])
def write_file():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    content = data.get('content', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400
    if len(content.encode('utf-8')) > MAX_FILE_SIZE:
        return jsonify({'status': 'error', 'error': 'Content too large to write'}), 400

    try:
        file_path = Path(path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding='utf-8')
        return jsonify({'status': 'success', 'path': str(file_path)})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/list_files', methods=['POST'])
def list_files():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        target = Path(path)
        if not target.exists():
            return jsonify({'status': 'error', 'error': 'Path not found'}), 404
        if not target.is_dir():
            return jsonify({'status': 'error', 'error': 'Path is not a directory'}), 400
        files = []
        for item in target.iterdir():
            label = item.name + '/' if item.is_dir() else item.name
            files.append(label)
        return jsonify({'status': 'success', 'files': files})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


def run_flask():
    app.run(host='0.0.0.0', port=PORT, use_reloader=False)


ngrok_token = os.environ.get('COLAB_NGROK_TOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

try:
    tunnel = ngrok.connect(PORT)
    print('Colab agent is running.')
    print(f'Public URL: {tunnel.public_url}')
    print(f"API Secret: {API_SECRET[:4]}{'*' * max(0, len(API_SECRET) - 4)}")
    print('Image models available: king2 (RASHID778/king2-image), mage-flow (microsoft/Mage-Flow)')
    print('Tip: use a GPU runtime (T4 or better) for image generation.')
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
except Exception as exc:
    print(f'Failed to start ngrok tunnel: {exc}')
    raise